In [ ]:
from pathlib import Path
from pprint import pprint

import numpy as np
import pandas as pd

import tensorflow as tf
from keras import Model, layers, models
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.optimizers import Adam

import matplotlib.pyplot as plt

In [ ]:
MAX_HEIGHT = 100.0
PATCH_SIZE = 128
PATCH_DEPTH = 11
NDVAL = -9999

TOPDIR = Path('/tmp/download')
PRED_dir = TOPDIR/'predict'
TFREC_dir = Path('/home/jovyan/ak_fire_tfrecs_rh98_11band')
MODEL_dir = Path('/home/jovyan/models')

TOPDIR.mkdir(exist_ok=True)
PRED_dir.mkdir(exist_ok=True)
TFREC_dir.mkdir(exist_ok=True)
MODEL_dir.mkdir(exist_ok=True)

# Model definition

In [ ]:
desc = {
    'height': tf.io.FixedLenFeature([], tf.int64),
    'width': tf.io.FixedLenFeature([], tf.int64),
    'depth': tf.io.FixedLenFeature([], tf.int64),
    'arr': tf.io.FixedLenFeature([], tf.string)
}

def _parse_image(ex):
    p_img = tf.io.parse_single_example(ex, desc)
    p_arr = tf.io.parse_tensor(p_img['arr'], out_type=tf.float32)
    p_arr.set_shape([128, 128, 11])
    X = p_arr[:,:,:-1]
    Y = p_arr[:,:,-1]
    return X, Y

def masked_mse_loss(mask_value=-9999):
    def loss(y_true, y_pred):
        mask = tf.cast(tf.not_equal(y_true, mask_value), tf.float32)
        squared_error = tf.square(y_true - y_pred) * mask        
        return tf.reduce_sum(squared_error) / (tf.reduce_sum(mask) + 1e-6)
    return loss

def conv_block(inputs, num_filters):
    x = layers.Conv2D(num_filters, 3, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(num_filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x

def encoder_block(inputs, num_filters):
    x = conv_block(inputs, num_filters)
    p = layers.MaxPooling2D((2, 2))(x)
    return x, p

def decoder_block(inputs, skip_features, num_filters):
    x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(inputs)
    x = layers.Concatenate()([x, skip_features])
    x = conv_block(x, num_filters)
    return x

def build_unet_4blocks(input_shape):
    inputs = layers.Input(input_shape)

    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    b1 = conv_block(p4, 1024)

    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    # outputs = layers.Conv2D(1, 1, padding="same", activation="relu")(d4)
    # model = Model(inputs, outputs, name="UNet_4blocks")

    probs = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(d4)    
    outputs = layers.Lambda(lambda x: x * MAX_HEIGHT, name="height_output")(probs)
    
    model = Model(inputs, outputs, name="UNet_4blocks")
    return model

# extract info from tfrecord filenames

In [ ]:
def num_records(tfrec_fn):
    return int(tfrec_fn.split('.')[0].split('_')[-1])

def tile_num(tfrec_fn):
    return tfrec_fn.split('/')[-1].split('_')[0]

def get_year(tfrec_fn):
    return tfrec_fn.split('/')[-1].split('_')[1]

def plot_(hist):
    plt.plot(hist.history["loss"], label="Train Loss")
    plt.plot(hist.history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.show()

# Plot a few the tfrecords

In [ ]:
tfrec_rnd_idx = np.random.randint(low=0, high=100, size=1)[0]
d = tf.data.TFRecordDataset(str(list(TFREC_dir.glob('*'))[tfrec_rnd_idx]), compression_type='GZIP')

plt.figure(figsize=(12, 100))
for i, rec in enumerate(d.map(_parse_image).take(20)):
    img = rec[0].numpy()[..., [2,1,0]] # true color
    #im = rec[0].numpy()[..., [2, 4]]
    # ndvi = im[...,1] - im[...,0] / im[...,1] + im[...,0]
    pmin, pmax = np.percentile(img, (2, 98))
    img = np.clip(img, pmin, pmax)
    img = (img - pmin) / (pmax - pmin)
    ax = plt.subplot(40,3,3*i+1)
    plt.imshow(img)
    plt.axis('off')

    img = rec[1].numpy()
    ax = plt.subplot(40,3,3*i+2)
    plt.imshow(img, cmap='viridis')
    plt.axis('off')
    
    img = rec[0].numpy()[...,8]
    ax = plt.subplot(40,3,3*i+3)
    plt.imshow(img, cmap='Reds')
    plt.axis('off')
    print(pmin, np.median(img), pmax)
plt.tight_layout(pad=0.5)


In [ ]:
tiles_around_3364 = [
    2994,2995,2996,2997,2998,2999,3000,3001,3082,
    3083,3084,3085,3086,3087,3088,3089,3090,3174,
    3175,3176,3177,3178,3179,3180,3181,3182,3267,
    3268,3269,3270,3271,3272,3273,3274,3275,3360,
    3361,3362,3363,3364,3365,3366,3367,3368,3453,
    3454,3455,3456,3457,3458,3459,3460,3461,3548,
    3549,3550,3551,3552,3553,3554,3555,3556,3642,
    3643,3644,3645,3646,3647,3648,3649,3650,3736,
    3737,3738,3739,3740,3741,3742,3743,3744,41056
]

tfrecs = [str(p) for p in TFREC_dir.glob('*.tfrecord.gz') if int(tile_num(str(p))) in tiles_around_3364 ]
lens = [int(x.split('_')[-1].split('.')[0]) for x in tfrecs]

print(sum(lens))
sorted(list(zip(tfrecs, lens)), key=lambda x: x[1], reverse=True)
print(f'{np.mean(lens)}, {np.median(lens)}, {np.quantile(lens, (0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1))}')

tfrec_df = pd.DataFrame([[int(tile_num(x)), get_year(x), x] for x in tfrecs], columns=['tile', 'year', 'path'])
pprint(tfrec_df.head())
tfrec_df.shape

np.random.seed(27182)
np.random.shuffle(tiles_around_3364)
split_idx = int(0.8 * len(tiles_around_3364))

td = tfrec_df[tfrec_df.tile.isin(tiles_around_3364[:split_idx])].sort_values(['tile', 'year']).groupby('tile').nth([0, 1, -1]).reset_index(drop=True)
num_train = td['path'].apply(lambda x: num_records(x)).astype(int).sum()

vd = tfrec_df[tfrec_df.tile.isin(tiles_around_3364[split_idx:])]
num_val = vd['path'].apply(lambda x: num_records(x)).astype(int).sum()

print(f'''number of training records: {num_train} 
number of validation records: {num_val},
train fraction: {num_train/(num_train+num_val)}''')

train_files = td.path.to_list()
np.random.shuffle(train_files)
pprint(train_files[:5])

val_files = vd.path.to_list()
np.random.shuffle(val_files)
pprint(val_files[:5])

In [ ]:
def prepare_ds(files, shuffle=False):
    ds = tf.data.TFRecordDataset(files, compression_type='GZIP')
    ds = ds.map(_parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000) 
    ds = ds.batch(16, drop_remainder=True)
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = prepare_ds(train_files, shuffle=False)
val_ds = prepare_ds(val_files, shuffle=False)

m4 = build_unet_4blocks(input_shape=(PATCH_SIZE, PATCH_SIZE, PATCH_DEPTH-1))
m4.compile(optimizer=Adam(learning_rate=0.001), loss=masked_mse_loss(NDVAL))
m4.summary()

h4 = m4.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    
    callbacks=[
        EarlyStopping(monitor='val_loss', start_from_epoch=10, verbose=1, patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3, verbose=1, min_lr=1e-6),  
        ModelCheckpoint(str(MODEL_dir/'m4_128_AK_fire_rh98_11bands.keras'), verbose=1, save_best_only=True)
    ]
)

plot_(h4)

In [ ]:
tf.keras.backend.clear_session()